In [ ]:
# Chunk X

# The goal for this script is to go from annotations to yolo-ready data
# Import your annotations


In [ ]:
# Chunk 0 - Auto-install missing packages
# Checks that all packages needed by this notebook are installed, and installs
# any that are missing before the imports below run. Safe to re-run.

import importlib
import subprocess
import sys

# Map: module name used in `import ...`  ->  package name used by pip install
REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "cv2": "opencv-python",
    "PIL": "Pillow",
    "datumaro": "datumaro",
    "yaml": "PyYAML",
    "sklearn": "scikit-learn",
    "tqdm": "tqdm",
}

def install_missing_packages(packages: dict) -> None:
    """
    Check whether each required package can be imported, and pip install
    any that are missing.

    Args:
        packages (dict): mapping of {import_name: pip_install_name}
    """
    missing = []
    for import_name, pip_name in packages.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            missing.append(pip_name)

    if missing:
        print(f"Installing missing packages: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
        print("Finished installing missing packages.")
    else:
        print("All required packages are already installed.")

install_missing_packages(REQUIRED_PACKAGES)


In [ ]:
# Chunk 0.1 - Core imports

# === Core Python ===
import os
import glob
import json
import random
import time
from collections import Counter, defaultdict
from typing import Dict, Any, List, Tuple
from pathlib import Path
import re
import shutil
# === Math & Data ===
import numpy as np
import pandas as pd

# === Visualization ===
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
from PIL import Image

# === Annotation / Dataset Handling ===
import datumaro as dm
from difflib import get_close_matches  # used by link_images_to_dataset()
from sklearn.model_selection import StratifiedGroupKFold
from tqdm import tqdm

import yaml




In [ ]:
# Chunk 1 - setup and imports

# If only using the testing-dataset used here, dont change anything. If youre adding your own dataset, add the path here
# This chunk checks that the paths are working and checking the structures of the .json file with the annotations
# and the number of classes

BASE_DIR = Path("/scratch/disk4/crab_model_gina/crabs-on-camera/data/annotations") # setting the base directory

CONFIG = {
    "ANNOTATIONS_FILE": BASE_DIR / "test_annotations/annotations/instances_default.json",
    "IMAGES_DIR": BASE_DIR / "test_annotations/images/default",
}

def get_dataset_paths() -> tuple:
    """Get full paths for the dataset and verify structure.

    Returns:
        tuple: (annotations_path, images_path, data)
    """
    annotations_path = CONFIG["ANNOTATIONS_FILE"]
    images_path = CONFIG["IMAGES_DIR"]

    if not annotations_path.exists():
        raise FileNotFoundError(f"Annotations file not found: {annotations_path}")
    if not images_path.exists():
        raise FileNotFoundError(f"Images directory not found: {images_path}")

    print(f"Using annotations: {annotations_path}")
    print(f"Using images: {images_path}")

    print("\nVerifying JSON structure...")
    with open(annotations_path, 'r') as f:
        data = json.load(f)
        print("Top-level keys:", list(data.keys()))

        if 'items' in data:
            print("\n✓ Has 'items' key (contains annotations)")
            print(f"Number of items: {len(data['items'])}")

            if len(data['items']) > 0:
                first_item = data['items'][0]
                print("\nFirst item structure:")
                print("Keys in first item:", list(first_item.keys()))

                if 'annotations' in first_item and first_item['annotations']:
                    print("\nFirst annotation structure:")
                    print("Keys in first annotation:", list(first_item['annotations'][0].keys()))
                    print("\nExample annotation:", json.dumps(first_item['annotations'][0], indent=2))
        else:
            print("Missing 'items' key")

        if 'categories' in data:
            print("\nHas 'categories' key")
            print(f"Number of categories: {len(data['categories'])}")
        else:
            print("Missing 'categories' key")

        if 'info' in data:
            print("\n Has 'info' key")

    return annotations_path, images_path, data


annotations_path, images_path, annotation_data = get_dataset_paths()

In [ ]:
# Chunk 2 - loading and validation
# This chunk validates and loads the dataset, and checks the first annotation
def load_and_validate_dataset() -> dm.Dataset:
    """
    Load and validate the dataset defined in CONFIG.

    Returns:
        dm.Dataset: the loaded Datumaro dataset
    """
    annotations_path = CONFIG["ANNOTATIONS_FILE"]
    images_path = CONFIG["IMAGES_DIR"]

    print("=" * 80)
    print("Loading dataset...")
    print(f"Annotations: {annotations_path}")
    print(f"Images: {images_path}")

    # Load raw JSON
    with open(annotations_path, "r") as f:
        data = json.load(f)

    # Basic validation
    if not isinstance(data.get("categories"), list):
        raise ValueError("JSON is not COCO-style (missing 'categories' list).")
    print("Detected COCO format.")

    # Load dataset using Datumaro
    dataset = dm.Dataset.import_from(str(annotations_path), "coco")
    print(f"Loaded {len(dataset)} items")

    # Label summary
    print("\nLabels:")
    for cat in data["categories"]:
        print(f"  {cat['id']}: {cat['name']}")

    # Find first annotated image
    first_annotated = next((i for i in dataset if len(i.annotations) > 0), None)
    if first_annotated:
        print(f"\nFirst annotated image: {first_annotated.id}")
        print(f"Total annotations: {len(first_annotated.annotations)}")
    else:
        print("Warning: no annotated items found!")

    print("=" * 80)
    print("Dataset loaded and validated successfully.")

    return dataset


dataset = load_and_validate_dataset()

In [ ]:
# Chunk 3 - converting RLE masks exported from CVAT in COCO 1.0 format into bounding boxes fitting a YOLO-format
# This also allows you to specify which species to import

def convert_masks_to_yolo_boxes(
    dataset: dm.Dataset,
    target_label: int = 1,          # 1 = Decapods
    target_species: str = "crab",   # Only keep Decapods with species=crab
    swap_dimensions: bool = False,  # Default = False (Datumaro gives (H, W))
    iou_threshold: float = 0.8
) -> dm.Dataset:
    """
    Convert true RLE masks to YOLO-style bounding boxes.
    Keeps only Decapods that have 'Species' = 'Crab' (case-insensitive).
    Corrects for Datumaro (H, W) ordering to avoid swapped dimensions.
    """

    def deduplicate_boxes(annotations, iou_threshold=0.8):
        """Remove duplicate boxes that heavily overlap, IoU > threshold."""
        unique, seen = [], []
        for ann in annotations:
            if not isinstance(ann, dm.Bbox):
                unique.append(ann)
                continue
            x, y, w, h = ann.x, ann.y, ann.w, ann.h
            box = np.array([x, y, x + w, y + h])
            duplicate = False
            for s in seen:
                xx1 = max(box[0], s[0])
                yy1 = max(box[1], s[1])
                xx2 = min(box[2], s[2])
                yy2 = min(box[3], s[3])
                inter = max(0, xx2 - xx1) * max(0, yy2 - yy1)
                union = (w * h) + ((s[2] - s[0]) * (s[3] - s[1])) - inter
                iou = inter / union if union > 0 else 0
                if iou > iou_threshold:
                    duplicate = True
                    break
            if not duplicate:
                seen.append(box)
                unique.append(ann)
        return unique

    print("=" * 100)
    print("Converting dataset (keeping only Decapod:Crab masks)...")

    start_time = time.time()
    total_items = len(dataset)
    converted_items, success, fail = [], 0, 0
    printed_example = False

    for item in dataset:
        converted_annotations = []

        # Datumaro returns (height, width)
        img_h, img_w = item.media.size

        # Optionally swap if dataset has reversed ordering
        if swap_dimensions:
            img_w, img_h = img_h, img_w

        for ann in item.annotations:
            if ann.type.name == "mask" and not hasattr(ann, "points"):
                # Filter: only target class
                if ann.label != target_label:
                    continue

                # Filter: species attribute contains target_species
                attrs = {k.lower(): str(v).lower() for k, v in ann.attributes.items()}
                species_attr = attrs.get("species", "")
                if target_species.lower() not in species_attr:
                    continue

                try:
                    x, y, w, h = ann.get_bbox()
                    if w <= 0 or h <= 0:
                        fail += 1
                        continue

                    converted_annotations.append(
                        dm.Bbox(x, y, w, h, label=ann.label, attributes=ann.attributes)
                    )
                    success += 1

                    # Print the first example for inspection
                    if not printed_example:
                        printed_example = True
                        x_center = (x + w / 2) / img_w
                        y_center = (y + h / 2) / img_h
                        w_norm = w / img_w
                        h_norm = h / img_h
                        print("\nExample conversion (first valid crab mask):")
                        print(f"   Image ID: {item.id}")
                        print(f"   Image size: {img_w}x{img_h}")
                        print(f"   Species: {species_attr}")
                        print(f"   Original bbox: [x={x:.1f}, y={y:.1f}, w={w:.1f}, h={h:.1f}]")
                        print(f"   YOLO normalized: "
                              f"(x={x_center:.4f}, y={y_center:.4f}, w={w_norm:.4f}, h={h_norm:.4f})")

                except Exception as e:
                    fail += 1
                    print(f"Error converting mask in {item.id}: {e}")

        # Deduplicate overlapping boxes
        before = len(converted_annotations)
        converted_annotations = deduplicate_boxes(converted_annotations, iou_threshold)
        after = len(converted_annotations)
        if after < before:
            print(f"Deduplicated {before - after} overlapping boxes in {item.id}")

        converted_items.append(item.wrap(annotations=converted_annotations, media=item.media))

    elapsed = time.time() - start_time
    print("Conversion complete")
    print(f"   Total images: {total_items:,}")
    print(f"   Boxes created (crabs only): {success:,}")
    print(f"   Failed masks: {fail:,}")
    print(f"   Time elapsed: {elapsed:.1f}s")
    print("=" * 100)

    return dm.Dataset.from_iterable(
        converted_items,
        categories=dataset.categories(),
        media_type=dm.Image
    )


converted_dataset = convert_masks_to_yolo_boxes(
    dataset,
    target_label=0,        # Decapods label
    target_species="crab",
    swap_dimensions=False  # keep False unless proven needed
)



In [ ]:
# Chunk 5 - Linking the images to the frames
def link_images_to_dataset(converted_dataset: dm.Dataset) -> dm.Dataset:
    """
    Link image files from CONFIG paths to the converted dataset.
    Uses fuzzy matching to handle minor filename differences.

    Args:
        converted_dataset (dm.Dataset): output from convert_masks_to_yolo_boxes()

    Returns:
        dm.Dataset: dataset with media attached
    """
    print("=" * 100)
    print("Linking images for dataset...")

    images_path = CONFIG["IMAGES_DIR"]
    print(f"Looking in: {images_path}")

    # Collect all available image files
    image_files = {
        os.path.splitext(f.lower())[0]: os.path.join(images_path, f)
        for f in os.listdir(images_path)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    }

    linked_items = []
    unmatched = []

    for item in converted_dataset:
        base_id = os.path.basename(item.id).lower()
        media = None

        # Try exact match
        if base_id in image_files:
            media = dm.Image.from_file(path=image_files[base_id])
        else:
            # Fuzzy match (handles small filename differences)
            match = get_close_matches(base_id, image_files.keys(), n=1, cutoff=0.7)
            if match:
                media = dm.Image.from_file(path=image_files[match[0]])
            else:
                unmatched.append(item.id)

        linked_items.append(item.wrap(media=media))

    # Build dataset with media attached
    linked_dataset = dm.Dataset.from_iterable(
        linked_items,
        categories=converted_dataset.categories(),
    )

    print("Linked dataset created.")
    print(f"   Total items: {len(linked_dataset)}")
    print(f"   Unmatched items: {len(unmatched)}")
    if unmatched:
        print(f"   Example unmatched: {unmatched[:5]}")

    print("=" * 100)
    print("Dataset linked with images successfully.")

    return linked_dataset


linked_dataset = link_images_to_dataset(converted_dataset)

In [ ]:

# Chunk 6 - Verifying and checking the first bbox conversion
def show_first_bbox(dataset: dm.Dataset) -> None:
    """
    Display the first annotated image from the dataset.
    Supports ann.bbox and ann.points formats.

    Args:
        dataset (dm.Dataset): a Datumaro dataset with linked media
    """
    print("=" * 100)
    print("Displaying first annotated image from dataset...")

    found = False
    for item in dataset:
        if len(item.annotations) > 0 and hasattr(item.media, "path") and item.media.path:
            print(f"Showing: {item.media.path}")
            img = Image.open(item.media.path)
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.imshow(img)

            for ann in item.annotations:
                # --- Handle ann.bbox format ---
                if hasattr(ann, "bbox") and ann.bbox:
                    x, y, w, h = ann.bbox
                    rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='r', facecolor='none')
                    ax.add_patch(rect)
                # --- Handle ann.points format (x_min, y_min, x_max, y_max) ---
                elif hasattr(ann, "points") and len(ann.points) >= 4:
                    x1, y1, x2, y2 = ann.points[:4]
                    w, h = x2 - x1, y2 - y1
                    rect = patches.Rectangle((x1, y1), w, h, linewidth=2, edgecolor='r', facecolor='none')
                    ax.add_patch(rect)

            plt.title(f"{os.path.basename(item.media.path)} — {len(item.annotations)} boxes")
            plt.axis("off")
            plt.show()
            found = True
            break  # stop after first image

    if not found:
        print("No valid image with annotations found in dataset.")

    print("=" * 100)
    print("Display complete.")


show_first_bbox(linked_dataset)



In [ ]:
# Chunk 6.1 - Comparing bbox to masks
def visualize_mask_vs_bbox(
    original_dataset: dm.Dataset,
    converted_dataset: dm.Dataset,
    sample_ids: list = None,
    target_labels: list = None,
    n_samples: int = 3,
):
    """
    Compare original RLE/polygon masks with converted bounding boxes.

    Args:
        original_dataset (dm.Dataset): raw RLE/polygon dataset
        converted_dataset (dm.Dataset): YOLO-style bbox dataset
        sample_ids (list, optional): Specific item IDs to visualize.
        target_labels (list, optional): Labels to visualize (default: all in dataset)
        n_samples (int): Number of random annotated samples to visualize.
    """
    print("=" * 120)
    print("Visualizing mask -> bbox comparison...")

    # Collect annotated items
    original_items = [item for item in original_dataset if len(item.annotations) > 0]
    if not original_items:
        print("No annotated items found in dataset.")
        return

    # Choose sample items
    if sample_ids is None:
        sample_items = random.sample(original_items, min(n_samples, len(original_items)))
    else:
        sample_items = [i for i in original_items if i.id in sample_ids]

    n = len(sample_items)
    fig, axes = plt.subplots(n, 2, figsize=(18, 6 * n))
    if n == 1:
        axes = np.array([axes])

    fig.suptitle("Mask -> Bounding Box Conversion Validation", fontsize=18, y=0.95)

    for idx, orig_item in enumerate(sample_items):
        conv_item = next((ci for ci in converted_dataset if ci.id == orig_item.id), None)
        if conv_item is None:
            print(f"No converted item found for ID: {orig_item.id}")
            continue

        # Load the image
        image_path = None
        if hasattr(orig_item.media, "path") and orig_item.media.path:
            image_path = orig_item.media.path
        elif hasattr(conv_item.media, "path") and conv_item.media.path:
            image_path = conv_item.media.path

        if not image_path or not os.path.exists(image_path):
            print(f"No image found for {orig_item.id}")
            continue

        image = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)

        # LEFT: original masks
        ax1 = axes[idx, 0]
        ax1.imshow(image)
        ax1.set_title(f"Original Masks — {os.path.basename(orig_item.id)}", fontsize=14)

        mask_count = 0
        for ann in orig_item.annotations:
            if target_labels is None or ann.label in target_labels:
                if ann.type.name == 'mask':
                    ax1.imshow(ann.image, alpha=0.4, cmap='cool')
                    mask_count += 1
        ax1.text(10, 30, f"{mask_count} masks", color='white', fontsize=12, backgroundcolor='black')
        ax1.axis("off")

        # RIGHT: converted bounding boxes
        ax2 = axes[idx, 1]
        ax2.imshow(image)
        ax2.set_title("Converted Bounding Boxes", fontsize=14)

        box_count = 0
        for ann in conv_item.annotations:
            if target_labels is None or ann.label in target_labels:
                if hasattr(ann, "bbox"):
                    x, y, w, h = ann.bbox
                elif hasattr(ann, "points") and len(ann.points) >= 4:
                    x1, y1, x2, y2 = ann.points[:4]
                    x, y, w, h = x1, y1, x2 - x1, y2 - y1
                else:
                    continue
                rect = plt.Rectangle((x, y), w, h, fill=False, edgecolor='red', linewidth=2)
                ax2.add_patch(rect)
                box_count += 1
        ax2.text(10, 30, f"{box_count} boxes", color='white', fontsize=12, backgroundcolor='black')
        ax2.axis("off")

    plt.tight_layout()
    plt.show()

    print("Completed visualization.\n")
    print("=" * 120)
    print("Mask -> BBox comparison done.")
    print("Validation Checklist:")
    print("1. Boxes should fully contain the mask area.")
    print("2. No duplicate boxes for the same object.")
    print("3. Object aspect ratios look biologically reasonable.")
    print("Remember, since we have removed non-crab annotations these are missing.")



visualize_mask_vs_bbox(
    original_dataset=dataset,               # your raw RLE/polygon dataset
    converted_dataset=converted_dataset,    # YOLO-style bbox dataset
    n_samples=3                             # number of examples to visualize
)

In [ ]:
# Chunk 7 - exporting all the converted annotatins to YOLO-format

## This function splits into 80/20 val, and makes sure the crab annotations are split evenly
## Creates a YAML, and debug output to check the splits + csv file with the same info
## Overlining the converted bbox and the exported bbox

def export_balanced_yolo_split_fixed(
    dataset: dm.Dataset,
    target_label: int,
    output_dir: str,
    val_ratio: float = 0.2,
    seed: int = 42,
    debug_examples: int = 3
):
    """
    Export Datumaro dataset to YOLO format with balanced train/val split.
    - Ensures even distribution of crab vs. empty images.
    - Overlays Datumaro vs. YOLO boxes for visual debugging.
    - Creates a single-class (crab) YOLO dataset.yaml.
    - Saves a train/val stats CSV for analysis in R.
    """

    # =====================================================
    # 1️⃣ Balanced split with detailed logging
    # =====================================================
    def split_dataset_balanced(dataset, target_label, val_ratio=0.2, seed=42):
        random.seed(seed)
        crab_items, non_crab_items = [], []

        for item in dataset:
            has_crab = any(
                ann.type.name == "bbox" and ann.label == target_label
                for ann in item.annotations
            )
            (crab_items if has_crab else non_crab_items).append(item)

        random.shuffle(crab_items)
        random.shuffle(non_crab_items)

        n_crabs = len(crab_items)
        n_val_crabs = int(val_ratio * n_crabs)
        n_noncrabs = len(non_crab_items)
        n_val_noncrabs = int(val_ratio * n_noncrabs)

        val_items = crab_items[:n_val_crabs] + non_crab_items[:n_val_noncrabs]
        train_items = crab_items[n_val_crabs:] + non_crab_items[n_val_noncrabs:]

        print("\n📊 Balanced Split Summary:")
        print(f"   Total items: {len(dataset):,}")
        print(f"   Crab items: {n_crabs:,}")
        print(f"   Non-crab items: {n_noncrabs:,}")
        print(f"   Validation set: {len(val_items):,} (≈{val_ratio*100:.1f}%)")
        print(f"   Train set: {len(train_items):,}")

        return train_items, val_items, n_crabs, n_noncrabs

    # =====================================================
    # 2️⃣ Perform split + initialize folders
    # =====================================================
    train_items, val_items, n_crabs, n_noncrabs = split_dataset_balanced(dataset, target_label, val_ratio, seed)
    os.makedirs(output_dir, exist_ok=True)
    printed_debug = 0
    split_stats = []  # 🟢 collect stats for CSV

    # =====================================================
    # 3️⃣ Export each split
    # =====================================================
    for split_name, items in [("train", train_items), ("val", val_items)]:
        print(f"\n📦 Exporting {split_name.upper()} split to YOLO format...")

        images_out = os.path.join(output_dir, "images", split_name)
        labels_out = os.path.join(output_dir, "labels", split_name)
        os.makedirs(images_out, exist_ok=True)
        os.makedirs(labels_out, exist_ok=True)

        total_boxes, total_images, empty_files = 0, 0, 0

        for item in items:
            if not hasattr(item.media, "path") or not item.media.path:
                continue

            img_path = item.media.path
            img_name = os.path.splitext(os.path.basename(img_path))[0]
            label_path = os.path.join(labels_out, f"{img_name}.txt")

            try:
                img_h, img_w = item.media.size  # Datumaro: (H, W)
            except Exception:
                continue

            yolo_lines = []
            for ann in item.annotations:
                # only export bboxes matching the target label (crab)
                if ann.type.name != "bbox" or ann.label != target_label:
                    continue

                x, y, w, h = ann.get_bbox()
                already_normalized = max(x, y, w, h) <= 1.5

                if already_normalized:
                    x_center, y_center, w_norm, h_norm = x, y, w, h
                else:
                    x_center = (x + w / 2) / img_w
                    y_center = (y + h / 2) / img_h
                    w_norm = w / img_w
                    h_norm = h / img_h

                # Clamp to [0, 1]
                x_center = np.clip(x_center, 0, 1)
                y_center = np.clip(y_center, 0, 1)
                w_norm = np.clip(w_norm, 0, 1)
                h_norm = np.clip(h_norm, 0, 1)

                # Always class 0 (crab)
                yolo_lines.append(f"0 {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")
                total_boxes += 1

                # 🧩 Optional visual check
                if printed_debug < debug_examples and split_name == "train":
                    print(f"\n🔍 Example annotation ({split_name}):")
                    print(f"   Image: {img_name}")
                    print(f"   Raw bbox: [x={x:.1f}, y={y:.1f}, w={w:.1f}, h={h:.1f}]")
                    print(f"   → YOLO: [x={x_center:.4f}, y={y_center:.4f}, w={w_norm:.4f}, h={h_norm:.4f}]")

                    img = cv2.imread(img_path)
                    if img is not None:
                        # 🟡 Datumaro bbox
                        cv2.rectangle(
                            img,
                            (int(x), int(y)),
                            (int(x + w), int(y + h)),
                            (0, 255, 255), 2
                        )
                        # 🔴 YOLO bbox
                        x1 = int((x_center - w_norm / 2) * img_w)
                        y1 = int((y_center - h_norm / 2) * img_h)
                        x2 = int((x_center + w_norm / 2) * img_w)
                        y2 = int((y_center + h_norm / 2) * img_h)
                        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)

                        plt.figure(figsize=(7, 6))
                        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                        plt.title(f"{split_name.upper()} Example {printed_debug+1}: {img_name}\n[Yellow] Datumaro | [Red] YOLO")
                        plt.axis("off")
                        plt.show()

                    printed_debug += 1

            # Write label file (even if empty)
            with open(label_path, "w") as f:
                if yolo_lines:
                    f.write("\n".join(yolo_lines))
                else:
                    empty_files += 1

            # Copy image
            shutil.copy2(img_path, os.path.join(images_out, os.path.basename(img_path)))
            total_images += 1

        print(f"   Images: {total_images:,} | Boxes: {total_boxes:,} | Empty files: {empty_files:,}")
        split_stats.append({
            "split": split_name,
            "total_images": total_images,
            "total_boxes": total_boxes,
            "empty_images": empty_files
        })

    # =====================================================
    # 4️⃣ Write single-class YOLO dataset YAML
    # =====================================================
    yaml_path = os.path.join(output_dir, "dataset.yaml")
    yaml_data = {
        "train": os.path.join(output_dir, "images/train"),
        "val": os.path.join(output_dir, "images/val"),
        "nc": 1,
        "names": ["crab"]
    }

    with open(yaml_path, "w") as f:
        yaml.dump(yaml_data, f, sort_keys=False)

    print(f"\n✅ YOLO dataset export complete (single-class).")
    print(f"   YAML saved at: {yaml_path}")
    print(f"   Structure:")
    print(f"   {output_dir}/images/[train|val]")
    print(f"   {output_dir}/labels/[train|val]")

    # =====================================================
    # 5️⃣ Save train/val distribution to CSV (one level up)
    # =====================================================
    def count_items(items):
        with_crab = sum(
            any(ann.type.name == "bbox" and ann.label == target_label for ann in item.annotations)
            for item in items
        )
        return with_crab, len(items) - with_crab

    train_crabs, train_empty = count_items(train_items)
    val_crabs, val_empty = count_items(val_items)

    print(f"\n📊 Train set: {train_crabs} crab / {train_empty} empty")
    print(f"📊 Val set:   {val_crabs} crab / {val_empty} empty")

    # Save CSV above output_dir
    base_dir = os.path.dirname(output_dir.rstrip("/"))
    csv_path = os.path.join(base_dir, "trainval_splits.csv")
    df = pd.DataFrame([
        {"split": "train", "crab_images": train_crabs, "empty_images": train_empty, "total_images": len(train_items)},
        {"split": "val", "crab_images": val_crabs, "empty_images": val_empty, "total_images": len(val_items)},
    ])
    df.to_csv(csv_path, index=False)

    print(f"\n📁 Stats CSV saved → {csv_path}")

    return yaml_path, train_items, val_items, csv_path





In [ ]:
yaml_path, train_items, val_items, csv_path = export_balanced_yolo_split_fixed(
    converted_datasets["training"],     # 👈 select the training dataset here
    target_label=0,                     # Decapod/Crab class index
    output_dir="/scratch/disk4/crab_model_gina/BRUV_Krabbe_modell/yolo_split",
    val_ratio=0.2,
    debug_examples=3
)



In [ ]:
# Chunk 7.1 - Location / BRUV station helpers for grouped, stratified splitting

LOCATION_PATTERN = re.compile(r"GarnRuse_([A-Za-z]+)_")
DEPLOYMENT_PATTERN = re.compile(r"_([A-Z]{2,3}\d{1,2})_(?:RA|B)\d?_")


def extract_location(image_id: str) -> str:
    """
    Extract the location name between 'GarnRuse_' and the next underscore.

    Example:
        'NO_2025_0729_GarnRuse_Esjaholmen_ES10_RA3_R6_frame_9' -> 'Esjaholmen'
    """
    match = LOCATION_PATTERN.search(image_id)
    return match.group(1) if match else "Unknown"


def extract_bruv_id(image_id: str) -> str:
    """
    Extract the BRUV deployment/station ID (e.g. 'ES10') from an image ID.
    This is the grouping key used to keep whole stations together in either
    train or val, so no station leaks across the split.

    Falls back to splitting on '_' and then to the location name if the
    regex doesn't match, so every image still gets grouped somewhere.
    """
    match = DEPLOYMENT_PATTERN.search(image_id)
    if match:
        return match.group(1)

    parts = image_id.split("_")
    if len(parts) > 5:
        return parts[5]

    return extract_location(image_id)


def list_unique_locations(dataset: dm.Dataset) -> None:
    """Print how many images come from each location."""
    locations = defaultdict(int)
    for item in dataset:
        locations[extract_location(item.id)] += 1

    print("=" * 80)
    print(f"Found {len(locations)} unique locations\n")
    for loc, count in sorted(locations.items(), key=lambda x: -x[1]):
        print(f"  {loc:<20} -> {count:>5} images")
    print("=" * 80)


def summarize_locations_by_species(dataset: dm.Dataset) -> None:
    """Print total / crab / empty image counts per location."""
    site_summary = defaultdict(lambda: {"total": 0, "crab": 0, "empty": 0})

    for item in dataset:
        loc = extract_location(item.id)
        site_summary[loc]["total"] += 1
        if len(item.annotations) > 0:
            site_summary[loc]["crab"] += 1
        else:
            site_summary[loc]["empty"] += 1

    print("=" * 90)
    print(f"Found {len(site_summary)} unique locations\n")
    print(f"{'Location':<20} {'Total':>7} {'Crab':>7} {'Empty':>7} {'% Crab':>8}")
    print("-" * 60)
    for loc, stats in sorted(site_summary.items(), key=lambda x: -x[1]["total"]):
        total, crab, empty = stats["total"], stats["crab"], stats["empty"]
        pct = (crab / total * 100) if total else 0
        print(f"{loc:<20} {total:7d} {crab:7d} {empty:7d} {pct:8.1f}%")
    print("=" * 90)


list_unique_locations(linked_dataset)
summarize_locations_by_species(linked_dataset)


In [ ]:
# Chunk 7.2 - Exporting to YOLO format as stratified, grouped K folds

## Splits into n_splits folds using StratifiedGroupKFold:
##  - Stratified on crab-present vs. empty, so each fold's train/val split
##    has a similar crab/empty ratio.
##  - Grouped by BRUV station (extract_bruv_id), so no station appears in
##    both train and val within the same fold (avoids leakage between
##    frames from the same camera deployment).
## Creates a dataset.yaml per fold, a master_dataset.yaml, and a
## fold_summary.csv with per-fold stats for a sanity check.

def export_yolo_stratified_group_kfold(
    dataset: dm.Dataset,
    target_label: int,
    output_dir: str,
    n_splits: int = 5,
    seed: int = 42,
) -> str:
    """
    Export a Datumaro dataset to YOLO format as stratified, grouped K folds.

    Args:
        dataset: linked Datumaro dataset (output of link_images_to_dataset()).
        target_label: annotation label id that marks a crab bbox.
        output_dir: root directory to write fold1/, fold2/, ... into.
        n_splits: number of folds.
        seed: random seed for the split.

    Returns:
        str: path to the master_dataset.yaml summarizing all folds.
    """
    print("=" * 100)
    print(f"Preparing {n_splits}-fold stratified group YOLO export -> {output_dir}")

    # --------------------------------------------------------------------
    # Build per-image metadata
    # --------------------------------------------------------------------
    records = []
    skipped = 0
    for item in dataset:
        if not hasattr(item.media, "path") or not item.media.path:
            skipped += 1
            continue

        has_crab = any(
            ann.type.name == "bbox" and ann.label == target_label
            for ann in item.annotations
        )
        records.append({
            "id": item.id,
            "item": item,
            "has_crab": int(has_crab),
            "location": extract_location(item.id),
            "bruv_id": extract_bruv_id(item.id),
        })

    if skipped:
        print(f"Skipped {skipped} items with no linked image file.")

    n_locations = len(set(r["location"] for r in records))
    n_stations = len(set(r["bruv_id"] for r in records))
    crab_ratio = 100 * np.mean([r["has_crab"] for r in records])
    print(f"Images: {len(records)} | Locations: {n_locations} | BRUV stations: {n_stations}")
    print(f"Images with crabs: {crab_ratio:.1f}%")
    print("=" * 100)

    if n_stations < n_splits:
        print(f"WARNING: only {n_stations} BRUV stations but {n_splits} folds requested "
              f"-- some folds may end up with very few/no stations for validation.")

    y = np.array([r["has_crab"] for r in records])
    groups = np.array([r["bruv_id"] for r in records])
    X = np.zeros(len(records))

    skf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    output_path = Path(output_dir)
    if output_path.exists():
        print(f"Removing existing folds in {output_dir}...")
        shutil.rmtree(output_path)
    output_path.mkdir(parents=True, exist_ok=True)

    fold_summaries = []

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y, groups), 1):
        print("-" * 100)
        print(f"Fold {fold_idx}/{n_splits}")

        fold_dir = output_path / f"fold{fold_idx}"
        img_train_dir = fold_dir / "images" / "train"
        img_val_dir = fold_dir / "images" / "val"
        lbl_train_dir = fold_dir / "labels" / "train"
        lbl_val_dir = fold_dir / "labels" / "val"
        for d in (img_train_dir, img_val_dir, lbl_train_dir, lbl_val_dir):
            d.mkdir(parents=True, exist_ok=True)

        train_records = [records[i] for i in train_idx]
        val_records = [records[i] for i in val_idx]

        def export_split(split_records, img_dest, lbl_dest, desc):
            n_boxes = 0
            n_empty = 0
            for r in tqdm(split_records, desc=desc, leave=False):
                item = r["item"]
                img_path = item.media.path
                img_h, img_w = item.media.size  # Datumaro: (H, W)

                yolo_lines = []
                for ann in item.annotations:
                    if ann.type.name != "bbox" or ann.label != target_label:
                        continue
                    bx, by, bw, bh = ann.get_bbox()
                    x_center = float(np.clip((bx + bw / 2) / img_w, 0, 1))
                    y_center = float(np.clip((by + bh / 2) / img_h, 0, 1))
                    w_norm = float(np.clip(bw / img_w, 0, 1))
                    h_norm = float(np.clip(bh / img_h, 0, 1))
                    yolo_lines.append(f"0 {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")
                    n_boxes += 1

                img_name = os.path.basename(img_path)
                stem = os.path.splitext(img_name)[0]

                with open(lbl_dest / f"{stem}.txt", "w") as f:
                    if yolo_lines:
                        f.write("\n".join(yolo_lines))
                    else:
                        n_empty += 1

                shutil.copy2(img_path, img_dest / img_name)

            return n_boxes, n_empty

        train_boxes, train_empty = export_split(train_records, img_train_dir, lbl_train_dir, f"Fold {fold_idx} train")
        val_boxes, val_empty = export_split(val_records, img_val_dir, lbl_val_dir, f"Fold {fold_idx} val")

        # Per-fold YAML
        yaml_path = fold_dir / "dataset.yaml"
        with open(yaml_path, "w") as f:
            yaml.dump({
                "path": str(fold_dir),
                "train": "images/train",
                "val": "images/val",
                "nc": 1,
                "names": ["crab"],
            }, f, sort_keys=False)

        # Sanity check: no BRUV station leakage between train/val
        train_stations = set(r["bruv_id"] for r in train_records)
        val_stations = set(r["bruv_id"] for r in val_records)
        overlap = train_stations & val_stations

        train_crab_pct = 100 * np.mean([r["has_crab"] for r in train_records]) if train_records else 0
        val_crab_pct = 100 * np.mean([r["has_crab"] for r in val_records]) if val_records else 0

        print(f"  Train: {len(train_records)} images ({train_crab_pct:.1f}% crab, {train_boxes} boxes, "
              f"{train_empty} empty) -- {len(train_stations)} stations")
        print(f"  Val:   {len(val_records)} images ({val_crab_pct:.1f}% crab, {val_boxes} boxes, "
              f"{val_empty} empty) -- {len(val_stations)} stations")
        if overlap:
            print(f"  WARNING: stations appear in both train and val: {overlap}")
        else:
            print("  No station leakage between train/val.")

        fold_summaries.append({
            "fold": fold_idx,
            "train_images": len(train_records),
            "val_images": len(val_records),
            "train_crab_pct": round(train_crab_pct, 1),
            "val_crab_pct": round(val_crab_pct, 1),
            "train_stations": len(train_stations),
            "val_stations": len(val_stations),
            "station_overlap": len(overlap),
        })

    # Master YAML + fold summary CSV
    master_yaml_path = output_path / "master_dataset.yaml"
    with open(master_yaml_path, "w") as f:
        f.write(
            f"# {n_splits}-fold cross-validation YOLO export\n"
            f"# Use fold<N>/dataset.yaml to train on a specific fold, e.g.:\n"
            f"#   yolo train data=fold1/dataset.yaml ...\n"
            f"nc: 1\n"
            f"names:\n"
            f"  0: crab\n"
        )

    summary_csv_path = output_path / "fold_summary.csv"
    pd.DataFrame(fold_summaries).to_csv(summary_csv_path, index=False)

    print("=" * 100)
    print(f"K-fold YOLO export complete -> {output_dir}")
    print(f"Master YAML: {master_yaml_path}")
    print(f"Fold summary CSV: {summary_csv_path}")
    print("=" * 100)

    return str(master_yaml_path)


In [ ]:
# Chunk 7.3 - Run the stratified group K-fold export

master_yaml_path = export_yolo_stratified_group_kfold(
    linked_dataset,                     # dataset with images linked (Chunk 5)
    target_label=0,                     # Decapod/Crab class index
    output_dir="/scratch/disk4/crab_model_gina/BRUV_Krabbe_modell/yolo_kfold",
    n_splits=5,
    seed=42,
)


In [ ]:
def get_yolo_dataset_statistics(yolo_dir: str):
    """
    Returns statistics about an exported YOLO dataset:
    - Number of frames with annotations per split
    - Number of empty frames per split
    - Species annotation counts per split
    """
    
    import os
    import pandas as pd
    from pathlib import Path
    
    yolo_path = Path(yolo_dir)
    
    # Load class names from YAML
    yaml_path = yolo_path / "dataset.yaml"
    with open(yaml_path, 'r') as f:
        import yaml
        config = yaml.safe_load(f)
    
    class_names = config['names']
    print(f"🏷️ Classes: {class_names}")
    
    stats = {}
    
    for split in ['train', 'val']:
        images_dir = yolo_path / "images" / split
        labels_dir = yolo_path / "labels" / split
        
        if not images_dir.exists() or not labels_dir.exists():
            print(f"❌ Missing directory for {split} split")
            continue
        
        # Count images and labels
        image_files = list(images_dir.glob("*.*"))
        label_files = list(labels_dir.glob("*.txt"))
        
        frames_with_annotations = 0
        empty_frames = 0
        species_counts = {i: 0 for i in range(len(class_names))}
        
        for label_file in label_files:
            with open(label_file, 'r') as f:
                lines = f.readlines()
            
            if lines and any(line.strip() for line in lines):
                frames_with_annotations += 1
                # Count annotations per class
                for line in lines:
                    line = line.strip()
                    if line:
                        parts = line.split()
                        if len(parts) >= 5:
                            class_id = int(parts[0])
                            if class_id in species_counts:
                                species_counts[class_id] += 1
            else:
                empty_frames += 1
        
        total_frames = len(image_files)
        
        print(f"\n📊 {split.upper()} SPLIT STATISTICS:")
        print(f"   Total frames: {total_frames:,}")
        print(f"   Frames with annotations: {frames_with_annotations:,} ({frames_with_annotations/total_frames*100:.1f}%)")
        print(f"   Empty frames: {empty_frames:,} ({empty_frames/total_frames*100:.1f}%)")
        print(f"   Species annotations:")
        for class_id, count in sorted(species_counts.items()):
            if count > 0:
                class_name = class_names[class_id]
                print(f"     {class_name}: {count:,} boxes")
        
        stats[split] = {
            'total_frames': total_frames,
            'frames_with_annotations': frames_with_annotations,
            'empty_frames': empty_frames,
            'species_counts': species_counts
        }
    
    # Print summary comparison
    print("\n" + "="*60)
    print("📈 SPLIT COMPARISON SUMMARY:")
    print("="*60)
    
    for class_id, class_name in enumerate(class_names):
        train_count = stats['train']['species_counts'][class_id]
        val_count = stats['val']['species_counts'][class_id]
        total_count = train_count + val_count
        if total_count > 0:
            train_pct = (train_count / total_count) * 100
            val_pct = (val_count / total_count) * 100
            print(f"   {class_name:15} Train: {train_count:4d} ({train_pct:5.1f}%) | Val: {val_count:4d} ({val_pct:5.1f}%)")
    
    train_empty = stats['train']['empty_frames']
    val_empty = stats['val']['empty_frames']
    total_empty = train_empty + val_empty
    train_empty_pct = (train_empty / total_empty) * 100 if total_empty > 0 else 0
    val_empty_pct = (val_empty / total_empty) * 100 if total_empty > 0 else 0
    print(f"   {'Empty':15} Train: {train_empty:4d} ({train_empty_pct:5.1f}%) | Val: {val_empty:4d} ({val_empty_pct:5.1f}%)")
    
    return stats

# Usage for your exported dataset:
print("🔍 Analyzing exported YOLO dataset...")
yolo_stats = get_yolo_dataset_statistics("/scratch/disk4/crab_model_gina/BRUV_Krabbe_modell/yolo_split")

In [ ]:
# Chunk 8 - debug step to check the masks vs. yolo export

def visualize_mask_vs_yolo_for_all_datasets(
    original_datasets: dict,
    yolo_root: str,
    sample_ids: list = None,
    n_samples: int = 3,
    splits: list = ["train", "val"],
    target_labels: list = None,
):
    """
    Compare original RLE/polygon masks with final YOLO exported bounding boxes.

    Args:
        original_datasets (dict): {'training': dm.Dataset, 'testing': dm.Dataset}
        yolo_root (str): Path to YOLO export root (containing images/labels folders)
        sample_ids (list, optional): Specific image IDs to visualize
        n_samples (int): Number of random annotated samples per split
        splits (list): Which YOLO splits to visualize ('train', 'val')
        target_labels (list, optional): Filter by label indices
    """

    for dataset_name, dataset in original_datasets.items():
        print("=" * 120)
        print(f"🎨 Visualizing MASK → YOLO comparison for {dataset_name.upper()} dataset...")

        annotated_items = [item for item in dataset if len(item.annotations) > 0]
        if not annotated_items:
            print(f"⚠️ No annotated items found in {dataset_name}.")
            continue

        # Pick random or specified samples
        if sample_ids:
            sample_items = [i for i in annotated_items if i.id in sample_ids]
        else:
            sample_items = random.sample(annotated_items, min(n_samples, len(annotated_items)))

        for split in splits:
            print(f"\n📦 Split: {split.upper()}")
            fig, axes = plt.subplots(len(sample_items), 2, figsize=(18, 6 * len(sample_items)))
            if len(sample_items) == 1:
                axes = np.array([axes])

            fig.suptitle(f"{dataset_name.upper()} — Mask → YOLO Box Validation ({split})", fontsize=18, y=0.95)

            for idx, item in enumerate(sample_items):
                img_path = item.media.path if hasattr(item.media, "path") else None
                if not img_path or not os.path.exists(img_path):
                    print(f"⚠️ Image not found for {item.id}")
                    continue

                image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
                img_h, img_w = image.shape[:2]

                # === LEFT: Original masks ===
                ax1 = axes[idx, 0]
                ax1.imshow(image)
                ax1.set_title(f"Original Masks — {os.path.basename(item.id)}", fontsize=14)
                mask_count = 0
                for ann in item.annotations:
                    if target_labels is None or ann.label in target_labels:
                        if ann.type.name == "mask" and getattr(ann, "image", None) is not None:
                            ax1.imshow(ann.image, alpha=0.4, cmap="cool")
                            mask_count += 1
                ax1.text(10, 30, f"{mask_count} masks", color="white", fontsize=12, backgroundcolor="black")
                ax1.axis("off")

                # === RIGHT: YOLO boxes ===
                ax2 = axes[idx, 1]
                ax2.imshow(image)
                ax2.set_title("YOLO Bounding Boxes", fontsize=14)

                label_filename = os.path.splitext(os.path.basename(img_path))[0] + ".txt"
                label_path = os.path.join(yolo_root, "labels", split, label_filename)

                if os.path.exists(label_path):
                    with open(label_path, "r") as f:
                        lines = f.readlines()

                    box_count = 0
                    for line in lines:
                        parts = line.strip().split()
                        if len(parts) != 5:
                            continue
                        cls, x_center, y_center, w, h = map(float, parts)
                        if target_labels and int(cls) not in target_labels:
                            continue

                        # Convert YOLO (normalized) → pixel coordinates
                        x = int((x_center - w / 2) * img_w)
                        y = int((y_center - h / 2) * img_h)
                        w_px = int(w * img_w)
                        h_px = int(h * img_h)

                        rect = plt.Rectangle((x, y), w_px, h_px, fill=False, edgecolor="red", linewidth=2)
                        ax2.add_patch(rect)
                        ax2.text(x, y - 5, f"cls {int(cls)}", color="red", fontsize=10, weight="bold")
                        box_count += 1

                    ax2.text(10, 30, f"{box_count} boxes", color="white", fontsize=12, backgroundcolor="black")
                else:
                    ax2.text(10, 30, "No YOLO labels found", color="white", fontsize=12, backgroundcolor="black")

                ax2.axis("off")

            plt.tight_layout()
            plt.show()

            print(f"✅ Completed visualization for split: {split.upper()}")

    print("=" * 120)
    print("🎯 Mask → YOLO bounding box comparison complete.")
    print("Checklist:")
    print("1️⃣ Boxes fully contain the mask area.")
    print("2️⃣ No missing boxes where masks exist.")
    print("3️⃣ Box scale and aspect ratio match object size.")


In [ ]:
visualize_mask_vs_yolo_for_all_datasets(
    original_datasets=datasets,  # your raw RLE/polygon Datumaro datasets
    yolo_root="/scratch/disk4/crab_model_gina/BRUV_Krabbe_modell/yolo_split",  # your YOLO export folder
    n_samples=3,                 # number of examples per dataset
    splits=["train", "val"],     # visualize both splits
    target_labels=[0]            # optional: restrict to the crab class
)



In [ ]:
# Chunk 8.1 Another debug checking random images in the exported dataset

def show_random_yolo_images(yolo_dir: str, split: str = "train", n: int = 5, seed: int = 0):
    """
    Visualize random YOLO images using OpenCV (same convention as Datumaro + mask visualizer)
    Ensures coordinates match the YOLO normalization standard exactly.
    """
    rng = random.Random(seed)

    yaml_path = os.path.join(yolo_dir, "dataset.yaml")
    if not os.path.exists(yaml_path):
        print(f"❌ dataset.yaml not found in {yolo_dir}")
        return

    with open(yaml_path, "r") as f:
        data_yaml = yaml.safe_load(f)

    images_dir = data_yaml["train"] if split == "train" else data_yaml["val"]
    labels_dir = images_dir.replace("images", "labels")

    names = data_yaml.get("names")
    if isinstance(names, dict):
        id2name = {int(k): v for k, v in names.items()}
    else:
        id2name = {i: n for i, n in enumerate(names or [])}

    image_files = [f for f in os.listdir(images_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    if not image_files:
        print(f"⚠️ No images found in {images_dir}")
        return

    samples = rng.sample(image_files, min(n, len(image_files)))
    print(f"🎲 Showing {len(samples)} random {split.upper()} images from {images_dir}\n")

    for img_file in samples:
        img_path = os.path.join(images_dir, img_file)
        label_path = os.path.join(labels_dir, os.path.splitext(img_file)[0] + ".txt")

        image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        img_h, img_w = image.shape[:2]

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.imshow(image)

        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                lines = f.readlines()

            for line in lines:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls, x_center, y_center, w, h = map(float, parts)

                # Convert YOLO (normalized center) → pixel-space top-left
                x1 = int((x_center - w / 2) * img_w)
                y1 = int((y_center - h / 2) * img_h)
                w_px = int(w * img_w)
                h_px = int(h * img_h)

                rect = patches.Rectangle((x1, y1), w_px, h_px, linewidth=2, edgecolor='red', facecolor='none')
                ax.add_patch(rect)

                cls_name = id2name.get(int(cls), str(int(cls)))
                ax.text(
                    x1,
                    max(0, y1 - 5),
                    cls_name,
                    color='yellow',
                    fontsize=10,
                    fontweight='bold',
                    bbox=dict(facecolor='black', alpha=0.4, edgecolor='none', pad=1),
                )
        else:
            ax.text(10, 30, "No YOLO labels found", color='white', fontsize=12, backgroundcolor='black')

        ax.set_title(f"{split.upper()} — {img_file}", fontsize=14)
        ax.axis("off")
        plt.tight_layout()
        plt.show()
        
yolo_dir = "/scratch/disk4/crab_model_gina/BRUV_Krabbe_modell/yolo_split"

show_random_yolo_images(yolo_dir, split="train", n=5)
show_random_yolo_images(yolo_dir, split="val", n=5)



In [ ]:
# Chunk 8.2 - Checking the bounds of the exports

def verify_yolo_label_bounds(labels_dir: str) -> bool:
    """
    Verify that all YOLO label files in a directory contain coordinates within [0, 1].
    Prints summary of any out-of-bound values.

    Args:
        labels_dir (str): Path to YOLO labels folder (e.g. '.../labels/train')

    Returns:
        bool: True if all labels are valid, False if any out-of-bounds values found.
    """
    total_files = 0
    total_boxes = 0
    invalid_boxes = 0
    invalid_files = []

    # Walk through all .txt label files
    for root, _, files in os.walk(labels_dir):
        for fname in files:
            if not fname.endswith(".txt"):
                continue

            total_files += 1
            path = os.path.join(root, fname)

            try:
                with open(path, "r") as f:
                    lines = f.readlines()

                for line_num, line in enumerate(lines, start=1):
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue  # skip invalid lines
                    _, x, y, w, h = map(float, parts[:5])
                    total_boxes += 1

                    if not (0 <= x <= 1 and 0 <= y <= 1 and 0 <= w <= 1 and 0 <= h <= 1):
                        invalid_boxes += 1
                        invalid_files.append((path, line_num, (x, y, w, h)))

            except Exception as e:
                print(f"⚠️ Could not read {path}: {e}")

    print("=" * 80)
    print(f"📂 Scanned {total_files} label files with {total_boxes} boxes.")
    if invalid_boxes == 0:
        print("✅ All bounding boxes are within [0, 1]. Your YOLO labels are valid!")
        return True
    else:
        print(f"❌ Found {invalid_boxes} out-of-bound boxes in {len(set(f for f, _, _ in invalid_files))} files:")
        for path, line_num, (x, y, w, h) in invalid_files[:10]:  # show only first 10
            print(f"  - {path} (line {line_num}): x={x:.3f}, y={y:.3f}, w={w:.3f}, h={h:.3f}")
        if invalid_boxes > 10:
            print(f"  ...and {invalid_boxes - 10} more.")
        return False
    



In [ ]:
# Chunk 8.3 - Verifying there are only crab annotation in the exported set
def verify_only_crabs(labels_root):
    """
    Recursively checks all YOLO label files under labels_root to ensure only class 0 (Crab) is present.
    """
    bad = []
    total_files = 0
    total_boxes = 0

    for file in glob.glob(os.path.join(labels_root, "**", "*.txt"), recursive=True):
        total_files += 1
        with open(file, "r") as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                total_boxes += 1
                try:
                    cls = int(parts[0])
                    if cls != 0:
                        bad.append((file, cls))
                except ValueError:
                    bad.append((file, "non-integer"))

    print(f"🔎 Scanned {total_files:,} label files containing {total_boxes:,} boxes.")

    if bad:
        print(f"❌ Found {len(bad):,} boxes with invalid class IDs (not 0).")
        print("Example problematic entries:")
        for f, cls in bad[:10]:
            print(f"  {os.path.basename(f)} → class {cls}")
    else:
        print("✅ All labels are class 0 (Crab only). Clean and ready for training!")

In [ ]:
# Label directories
train_labels = "/scratch/disk4/crab_model_gina/BRUV_Krabbe_modell/yolo_split/labels/train"
val_labels   = "/scratch/disk4/crab_model_gina/BRUV_Krabbe_modell/yolo_split/labels/val"

# 1️⃣ Verify normalization
verify_yolo_label_bounds(train_labels)
verify_yolo_label_bounds(val_labels)

# 2️⃣ Verify only crabs (class 0)
verify_only_crabs(train_labels)
verify_only_crabs(val_labels)